# Fine-Tuning Leve — Classificador de Qualidade de Perguntas (Stack Overflow)

**Case study construído sobre o repositório de referência do curso** [Generative AI for Data Science](https://github.com/dgodevais/coursera-genai-for-data-science) (Microsoft/Coursera).

## Decisão técnica (documentada por transparência)

O notebook original (`fine-tune-demo.ipynb`) faz **QLoRA fine-tuning do Mistral-7B-Instruct**
(7 bilhões de parâmetros, quantizado em 4-bit, `device_map="cuda:0"`) — **exige GPU dedicada**.

Este case adapta a mesma tarefa (classificar perguntas do Stack Overflow em `HQ` / `LQ_EDIT` / `LQ_CLOSE`)
pra rodar **de graça, em CPU, dentro de um GitHub Codespace**, trocando:

| Original | Este case | Por quê |
|---|---|---|
| Mistral-7B-Instruct (causal LM, geração + prompt) | `distilbert-base-uncased` (encoder, classificação nativa) | Classificação é o encaixe certo pra um encoder — mais rápido, mais barato, e **sem perda de qualidade pra essa tarefa específica** |
| QLoRA 4-bit + `bitsandbytes` (exige CUDA) | Fine-tuning completo em fp32 (66M parâmetros cabe em CPU) | Sem dependência de GPU |
| `peft` + `accelerate` | `transformers.Trainer` puro | Menos dependências, mais fácil de reproduzir |

Isso não é "a versão fraca" do original — é a **escolha de engenharia certa pra um problema de classificação**,
documentada como trade-off consciente (é exatamente o tipo de decisão que separa quem só rodou um tutorial
de quem entende o motivo de cada peça).

> ⚠️ Este notebook precisa baixar o checkpoint `distilbert-base-uncased` do Hugging Face Hub —
> requer internet aberta (funciona no Codespaces; pode falhar em sandboxes com rede restrita).

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)

from data_prep import load_and_clean, split_dataframe, LABELS, LABEL2ID, ID2LABEL

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1. Carregar e limpar os dados reais

Mesmo `train.csv` do repositório original (perguntas do Stack Overflow rotuladas HQ / LQ_EDIT / LQ_CLOSE,
fonte: [60k Stack Overflow Questions](https://paperswithcode.com/dataset/60k-stack-overflow-questions)).

Achado real na limpeza: **1.1% das linhas têm o rótulo `Y` nulo** — não documentado no repo original.
Como é a própria variável-alvo, não dá pra imputar; descartamos essas linhas.

In [ ]:
MAX_ROWS = None  # None = usa o dataset inteiro (~29k linhas, roda tranquilo em CPU)

df = load_and_clean("../train.csv", max_rows=MAX_ROWS)
train_df, val_df, test_df = split_dataframe(df)  # mesma proporção 60/20/20 do notebook original

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
df["Y"].value_counts(normalize=True)

## 2. Tokenização

DistilBERT usa um encoder bidirecional — não precisamos do prompt estilo `[INST]...[/INST]`
usado pra "convencer" um LLM causal a responder em formato de classificação. O texto de entrada
já é (título + corpo + tags), e o modelo aprende a mapear direto pra uma das 3 classes.

In [ ]:
MODEL_ID = "distilbert-base-uncased"
MAX_TOKEN_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_TOKEN_LEN)

def to_hf_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]])
    return ds.map(tokenize, batched=True)

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 3. Fine-tuning

Hiperparâmetros mais leves que o original (batch maior, menos épocas) — 66M parâmetros em fp32
converge rápido para esta tarefa; não precisamos de 4 épocas como o Mistral-7B com LoRA precisava.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
).to(device)

BATCH_SIZE = 16
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5

training_args = TrainingArguments(
    output_dir="../models/checkpoints",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {"accuracy": accuracy_score(labels, preds), "precision_macro": precision,
            "recall_macro": recall, "f1_macro": f1}

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=data_collator, compute_metrics=compute_metrics,
)

trainer.train()

## 4. Avaliação no holdout de teste

Mesmas métricas do notebook original, pra comparação direta: Acurácia, Precisão Macro, Recall Macro, F1 Macro e F1 Micro.

In [ ]:
test_results = trainer.evaluate(test_ds)
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = test_df["label"].to_numpy()

precision, recall, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
_, _, f1_micro, _ = precision_recall_fscore_support(y_true, y_pred, average="micro")

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision Macro: {precision:.4f}")
print(f"Recall Macro: {recall:.4f}")
print(f"F1 Score Macro: {f1_macro:.4f}")
print(f"F1 Score Micro: {f1_micro:.4f}")

## 5. Salvar o modelo fine-tunado

Vamos usar esse checkpoint no próximo notebook (`02_quantize_and_benchmark.ipynb`) pra medir
o ganho real de tamanho/latência da quantização — comparação justa, mesmo modelo, mesma máquina.

In [ ]:
model.save_pretrained("../models/distilbert-so-quality-ft")
tokenizer.save_pretrained("../models/distilbert-so-quality-ft")
print("Modelo salvo em ../models/distilbert-so-quality-ft")